# Importacion de los archivos ```indice_marginacion_municipal_1990_2015``` y ```indice_marginacion_municipal_2020``` 
1. Configuracion e imports
2. Cargar ambosa rchivos excel
3. Inspeccion rapida de los datos
4. Filtrar Hermosillo en cada DF por separado
5. Limpiar tipos
6. Homologar columans con un diccionario
7. Unir con pd.concat
8. Validad resultado
9. Guardar el DF unificado en data/processed


# 1. Configuración y rutas

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Funcion para encontrar la raíz del proyecto
def encontrar_raiz(inicio=None) -> Path: # Path significa que la función devuelve un objeto de tipo Path
    """Sube por el árbol de carpetas hasta encontrar la raíz del proyecto."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        tiene_data = (carpeta / "data").is_dir()
        tiene_marca = any((carpeta / m).exists() for m in (".git", "Makefile", "pyproject.toml"))
        if tiene_data and tiene_marca:
            return carpeta
    raise FileNotFoundError("No encontré la raíz del proyecto.")


RAIZ = encontrar_raiz()

# Carpetas de entrada y salida (relativas a la raíz del proyecto)
DIR_RAW = RAIZ / "data" / "raw"
DIR_INTERIM = RAIZ / "data" / "interim"
DIR_INTERIM.mkdir(parents=True, exist_ok=True)

# Archivos de entrada. Para 2020 usamos glob porque la extensión puede ser .xls o .xlsx
ARCHIVO_1990_2015 = DIR_RAW / "indice_marginacion_municipal_1990_2015.xlsx"
ARCHIVO_2020 = next(DIR_RAW.glob("indice_marginacion_municipal_2020.*"))

# Hojas que contienen los datos (ignoramos la hoja del diccionario)
HOJA_1990_2015 = "Base_Indice_de_marginacion_muni"
HOJA_2020 = "IMM_2020"

# Clave INEGI de Hermosillo: entidad 26 (Sonora) + municipio 030 = 26030
CVE_MUN_HERMOSILLO = 26030

# Archivo de salida
ARCHIVO_SALIDA = DIR_INTERIM / "indice_marginacion_hermosillo_1990_2020.csv"

print("Raíz:", RAIZ)
print("Entrada 1:", ARCHIVO_1990_2015.name)
print("Entrada 2:", ARCHIVO_2020.name)
print("Salida:", ARCHIVO_SALIDA)

Raíz: C:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda
Entrada 1: indice_marginacion_municipal_1990_2015.xlsx
Entrada 2: indice_marginacion_municipal_2020.xlsx
Salida: C:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda\data\interim\indice_marginacion_hermosillo_1990_2020.csv


# 2. Cargar los excel

In [2]:
df_1990_2015 = pd.read_excel(ARCHIVO_1990_2015, sheet_name=HOJA_1990_2015)
df_2020 = pd.read_excel(ARCHIVO_2020, sheet_name=HOJA_2020)

print("1990-2015:", df_1990_2015.shape)
print("2020:", df_2020.shape)

1990-2015: (14646, 24)
2020: (72, 18)


# 3. Inspección rápida (tipos y nulos)

In [3]:
for nombre, df in [("1990-2015", df_1990_2015), ("2020", df_2020)]:
    print("=" * 70)
    print(nombre)
    resumen = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "ejemplo": df.iloc[0],
    })
    display(resumen)

1990-2015


,dtype,nulos,ejemplo
CVE_MUN,object,0,1001
CVE_ENT,object,0,1
ENT,str,0,Aguascalientes
CVEE_MUN,object,0,1
MUN,str,0,Aguascalientes
POB_TOT,object,0,506 274
VP,object,0,-
ANALF,object,0,6.05
SPRIM,object,0,27.99
OVSDE,object,0,6.55


2020


,dtype,nulos,ejemplo
CVE_ENT,int64,0,26
NOM_ENT,str,0,Sonora
CVE_MUN,int64,0,26001
NOM_MUN,str,0,Aconchi
POB_TOT,int64,0,2563
ANALF,float64,0,2.691722
SBASC,float64,0,37.125444
OVSDE,float64,0,0.780336
OVSEE,float64,0,0.858369
OVSAE,float64,0,0.702302


# 4. Filtrar Hermosillo en ambos dataframes
`CVE_MUN` puede venir como número o como texto (a veces con espacios). La función `a_numero` lo normaliza para que la comparación con 26030 no falle por tipos.

In [ ]:
# funcion para convertir una columna a numérica y quitar espacios y cambiar '-' por NaN 
def a_numero(serie: pd.Series) -> pd.Series:
    """Convierte una columna a numérica: quita espacios, cambia '-' por NaN."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    limpia = (
        serie.astype(str)
        .str.replace(r"\s+", "", regex=True)      # quita espacios (incluye separador de miles)
        .replace({"-": np.nan, "": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(limpia, errors="coerce")


def filtrar_municipio(df: pd.DataFrame, cve_mun: int) -> pd.DataFrame:
    """Devuelve solo las filas del municipio indicado (por clave de 5 dígitos)."""
    mascara = a_numero(df["CVE_MUN"]) == cve_mun
    return df[mascara].copy()   # .copy() evita advertencias al modificar después

# Filtrado de Hermosillo para 1990-2015 y 2020 usando la función filtrar_municipio
h_1990_2015 = filtrar_municipio(df_1990_2015, CVE_MUN_HERMOSILLO)
h_2020 = filtrar_municipio(df_2020, CVE_MUN_HERMOSILLO)

print("Filas Hermosillo 1990-2015:", len(h_1990_2015))
print("Años encontrados:", sorted(a_numero(h_1990_2015["AÑO"]).dropna().unique()))
print("Filas Hermosillo 2020:", len(h_2020))

if h_1990_2015.empty or h_2020.empty:
    raise ValueError("Alguno de los filtros quedó vacío. Revisa la clave y el tipo de CVE_MUN.")

display(h_1990_2015[["CVE_MUN", "MUN", "AÑO"]])
display(h_2020[["CVE_MUN", "NOM_MUN"]])

Filas Hermosillo 1990-2015: 6
Años encontrados: [np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015)]
Filas Hermosillo 2020: 1


,CVE_MUN,MUN,AÑO
1889,26030,Hermosillo,1990
4301,26030,Hermosillo,1995
6738,26030,Hermosillo,2000
9189,26030,Hermosillo,2005
11645,26030,Hermosillo ...,2010
14102,26030,Hermosillo,2015


,CVE_MUN,NOM_MUN
29,26030,Hermosillo


## 5. Limpiar tipos
Convertimos a número todas las columnas, excepto las de texto (nombres y grado de marginación).

In [11]:
def limpiar_tipos(df: pd.DataFrame, columnas_texto: list) -> pd.DataFrame:
    """Convierte a numéricas las columnas que no sean de texto y avisa si se pierden datos."""
    df = df.copy()
    for col in df.columns:
        if col in columnas_texto:
            df[col] = df[col].astype("string").str.strip()
        else:
            antes = df[col].notna().sum()       # valores no nulos antes de convertir
            df[col] = a_numero(df[col])
            despues = df[col].notna().sum()     # valores no nulos después
            # Perder datos al convertir sugiere una columna de texto que no está en la lista
            # (los '-' que pasan a NaN a propósito también cuentan; revisa si son esos)
            if despues < antes:
                print(f"Aviso: '{col}' perdió {antes - despues} valor(es) al convertir a número")
    return df

h_1990_2015 = limpiar_tipos(h_1990_2015, columnas_texto=["ENT", "MUN", "GM"])
h_2020 = limpiar_tipos(h_2020, columnas_texto=["NOM_ENT", "NOM_MUN", "GM_2020"])

display(h_1990_2015.dtypes.to_frame("dtype_1990_2015"))
display(h_2020.dtypes.to_frame("dtype_2020"))

,dtype_1990_2015
CVE_MUN,int64
CVE_ENT,int64
ENT,string
CVEE_MUN,int64
MUN,string
POB_TOT,int64
VP,float64
ANALF,float64
SPRIM,float64
OVSDE,float64


,dtype_2020
CVE_ENT,int64
NOM_ENT,string
CVE_MUN,int64
NOM_MUN,string
POB_TOT,int64
ANALF,float64
SBASC,float64
OVSDE,float64
OVSEE,float64
OVSAE,float64


## 6. Homologar columnas del archivo 2020

Renombramos las columnas de 2020 a los nombres del archivo 1990-2015 cuando representan lo mismo. `rename` ignora las llaves que no existan.

In [19]:
# --- 1. Equivalencias: nombre en 2020 -> nombre en 1990-2015 ---
# Solo las columnas cuyo nombre cambia; las que se llaman igual no necesitan renombrarse.
EQUIVALENCIAS_2020 = {
    "NOM_ENT": "ENT",
    "NOM_MUN": "MUN",
    "SBASC": "SPRIM",      # ⚠ revisar: ver nota 1 abajo
    "PL.5000": "PL<5000",
    "IM_2020": "IM",       # ⚠ revisar: ver nota 2 abajo
    "GM_2020": "GM",
}

# --- 2. Columnas a eliminar (existen solo en uno de los archivos) ---
ELIMINAR_1990_2015 = [
    "CVEE_MUN", "VP", "OVSD", "IND0A100", "LUGAR_NAC", "LUGAR_EST",
    "OVSDSE",              # no la mencionaste; solo existe en 1990-2015. Confirma.
]
ELIMINAR_2020 = [
    "IMN_2020", "POSICION",   # no las mencionaste; solo existen en 2020. Confirma.
]


def verificar_columnas(df: pd.DataFrame, columnas, nombre: str) -> None:
    """Falla con un mensaje claro si alguna columna indicada no existe en el DataFrame."""
    faltan = [c for c in columnas if c not in df.columns]
    assert not faltan, f"Estas columnas no existen en {nombre}: {faltan}"


# --- 3. Verificar que todo lo que escribimos existe (detecta errores de dedo) ---
verificar_columnas(h_2020, EQUIVALENCIAS_2020.keys(), "2020 (equivalencias)")
verificar_columnas(h_2020, ELIMINAR_2020, "2020 (eliminar)")
verificar_columnas(h_1990_2015, ELIMINAR_1990_2015, "1990-2015 (eliminar)")

# --- 4. Aplicar: renombrar y eliminar ---
h_2020 = h_2020.rename(columns=EQUIVALENCIAS_2020).drop(columns=ELIMINAR_2020)
h_1990_2015 = h_1990_2015.drop(columns=ELIMINAR_1990_2015)

# Evita duplicados: dos columnas distintas que terminen con el mismo nombre
assert not h_2020.columns.duplicated().any(), "Hay columnas duplicadas en 2020 tras renombrar"

# --- 5. Añadir año (2020 no lo trae) y la columna de procedencia ---
h_2020["AÑO"] = 2020
h_1990_2015["FUENTE"] = "1990_2015"
h_2020["FUENTE"] = "2020"

# --- 6. Resumen: ¿quedaron ambos DataFrames con las mismas columnas? ---
solo_1990 = sorted(set(h_1990_2015.columns) - set(h_2020.columns))
solo_2020 = sorted(set(h_2020.columns) - set(h_1990_2015.columns))
print("Columnas en 1990-2015:", len(h_1990_2015.columns))
print("Columnas en 2020:     ", len(h_2020.columns))
print("Solo en 1990-2015:", solo_1990)   # debe salir []
print("Solo en 2020:     ", solo_2020)   # debe salir []

Columnas en 1990-2015: 18
Columnas en 2020:      18
Solo en 1990-2015: []
Solo en 2020:      []


# 7. Unificación y validación

In [20]:
# ============================================================
# UNIFICAR (concat) + VALIDAR, todo en una celda
# ============================================================

COLUMNAS_TEXTO = ["ENT", "MUN", "GM", "FUENTE"]
ANIOS_ESPERADOS = [1990, 1995, 2000, 2005, 2010, 2015, 2020]

# --- 1. Pre-chequeo: ambos DataFrames deben tener exactamente las mismas columnas ---
assert set(h_1990_2015.columns) == set(h_2020.columns), (
    f"Columnas distintas. Solo en 1990-2015: {set(h_1990_2015.columns) - set(h_2020.columns)} | "
    f"Solo en 2020: {set(h_2020.columns) - set(h_1990_2015.columns)}"
)

# --- 2. Unir: apila las filas de 2020 debajo de las de 1990-2015 ---
# Se fuerza el mismo orden de columnas en ambos para que no haya desalineación.
columnas = h_1990_2015.columns.tolist()
df_hermosillo = pd.concat([h_1990_2015[columnas], h_2020[columnas]], ignore_index=True)

# Columnas de identificación al inicio; el resto en su orden original
primeras = ["CVE_ENT", "ENT", "CVE_MUN", "MUN", "AÑO", "FUENTE"]
resto = [c for c in df_hermosillo.columns if c not in primeras]
df_hermosillo = df_hermosillo[primeras + resto].sort_values("AÑO").reset_index(drop=True)

# --- 3. Validaciones (cada assert detiene la ejecución si algo falla) ---
# 3.1 Conteo de filas: lo que entró debe ser igual a lo que salió
assert len(df_hermosillo) == len(h_1990_2015) + len(h_2020), "Se perdieron o duplicaron filas en el concat"

# 3.2 Un solo municipio y la clave correcta
assert df_hermosillo["CVE_MUN"].nunique() == 1, "Hay más de un municipio"
assert df_hermosillo["CVE_MUN"].iloc[0] == CVE_MUN_HERMOSILLO, "La clave no es la de Hermosillo"

# 3.3 Años: sin duplicados y con los esperados
assert not df_hermosillo["AÑO"].duplicated().any(), "Hay años duplicados"
anios = df_hermosillo["AÑO"].tolist()
faltan = sorted(set(ANIOS_ESPERADOS) - set(anios))
sobran = sorted(set(anios) - set(ANIOS_ESPERADOS))
if faltan or sobran:
    print(f"⚠ Años faltantes: {faltan} | Años inesperados: {sobran}")

# 3.4 Tipos: todas las columnas que no son texto deben ser numéricas
no_numericas = [
    c for c in df_hermosillo.columns
    if c not in COLUMNAS_TEXTO and not pd.api.types.is_numeric_dtype(df_hermosillo[c])
]
assert not no_numericas, f"Columnas que deberían ser numéricas y no lo son: {no_numericas}"

# 3.5 Nombre de entidad y municipio consistentes
print("ENT:", df_hermosillo["ENT"].unique(), "| MUN:", df_hermosillo["MUN"].unique())

# --- 4. Resumen del resultado ---
print(f"\nShape final: {df_hermosillo.shape}")
print("Filas por fuente:")
print(df_hermosillo["FUENTE"].value_counts().to_string())
print("\n✔ Validaciones superadas")

# Tabla unificada y mapa de disponibilidad de datos (True = hay dato)
display(df_hermosillo)
display(df_hermosillo.set_index("AÑO").notna().T)

ENT: <ArrowStringArray>
['Sonora']
Length: 1, dtype: string | MUN: <ArrowStringArray>
['Hermosillo']
Length: 1, dtype: string

Shape final: (7, 18)
Filas por fuente:
FUENTE
1990_2015    6
2020         1

✔ Validaciones superadas


,CVE_ENT,ENT,CVE_MUN,MUN,AÑO,FUENTE,POB_TOT,ANALF,SPRIM,OVSDE,OVSEE,OVSAE,VHAC,OVPT,PL<5000,PO2SM,IM,GM
0,26,Sonora,26030,Hermosillo,1990,1990_2015,448966,3.68000,22.220000,4.820000,5.360000,4.35000,51.420000,13.850000,6.530000,41.880000,-1.980000,Muy bajo
1,26,Sonora,26030,Hermosillo,1995,1990_2015,559154,3.24000,NaN,NaN,3.650000,3.23000,NaN,NaN,NaN,NaN,-1.693000,Muy bajo
2,26,Sonora,26030,Hermosillo,2000,1990_2015,609829,3.01000,17.110000,NaN,1.540000,1.60000,39.080000,9.900000,6.790000,30.830000,-1.820000,Muy bajo
3,26,Sonora,26030,Hermosillo,2005,1990_2015,701838,2.49000,12.690000,0.890000,1.100000,2.27000,35.430000,6.910000,4.890000,23.760000,-1.781000,Muy bajo
4,26,Sonora,26030,Hermosillo,2010,1990_2015,784342,2.11000,10.800000,0.830000,0.720000,1.93000,31.270000,4.720000,4.130000,19.240000,-1.772000,Muy bajo
5,26,Sonora,26030,Hermosillo,2015,1990_2015,884273,1.29000,7.790000,0.570000,0.400000,1.58000,23.300000,1.650000,4.130000,19.950000,-1.728000,Muy bajo
6,26,Sonora,26030,Hermosillo,2020,2020,936263,1.43976,17.562774,0.187077,0.258912,0.48808,13.896215,1.791786,3.713914,47.174923,60.443754,Muy bajo


AÑO,1990,1995,2000,2005,2010,2015,2020
CVE_ENT,True,True,True,True,True,True,True
ENT,True,True,True,True,True,True,True
CVE_MUN,True,True,True,True,True,True,True
MUN,True,True,True,True,True,True,True
FUENTE,True,True,True,True,True,True,True
POB_TOT,True,True,True,True,True,True,True
ANALF,True,True,True,True,True,True,True
SPRIM,True,False,True,True,True,True,True
OVSDE,True,False,False,True,True,True,True
OVSEE,True,True,True,True,True,True,True


# Detalle:  

En 2020 CONAPO cambió la forma de calcular el índice. Sustituyó el procedimiento tradicional (componentes principales) por el método de distancia P2 (DP2). El objetivo declarado fue mejorar la comparabilidad en el tiempo y entre unidades geográficas. Eso explica las escalas distintas (-1.8 frente a 57) y la dirección contraria. 

En el IM de 2020, mayor valor significa menos marginación. En el archivo viejo es al revés: Asientos (-0.383, grado “Medio”) tiene un IM más alto que Aguascalientes (-1.833, “Muy bajo”). Si renombraste IM_2020 a IM, esa columna mezcla dos escalas con direcciones opuestas.